# Symbolic computation and gradients

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/symbolic_and_gradients.ipynb)

The inferences in `solver_tutorial.ipynb` are computed by evaluating a circuit over numbers. This notebook evaluates the same circuits over **symbolic expressions** with [SymPy](https://www.sympy.org/), which gives the exact equations behind each inference, and uses them to study **gradients** with respect to the weights. It covers:

1. **Symbolic expressions** for probabilities and the other semiring inferences.
2. **Gradients**: numerical (PyTorch), symbolic and Boolean, and what they mean.
3. **Learning weights** by gradient descent.

## Setup

We use the same formulas and weights as in `solver_tutorial.ipynb`, together with its brute-force truth table for checking results (uncomment the install line if you are running on Colab). If `symprog` is not installed, the notebook uses the version in this repository.

In [1]:
# %pip install -q "symprog[notebooks,jax]"

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

import itertools
import math

import sympy
import torch
from symprog import Solver

FORMULAS = [
    "Implies(a, b)",                 # a -> b
    "And(Or(a, b), Implies(a, c))",  # (a or b) and (a -> c)
]
WEIGHTS = {"a": 0.9, "b": 0.64, "c": 0.34}

solver = Solver()
solver.add(FORMULAS)
solver.add_weights(WEIGHTS)

## brute-force truth table (see solver_tutorial.ipynb)
VARIABLES = ["a", "b", "c"]

def F(a, b, c):
    """FORMULAS as a Python function: (a -> b) and (a or b) and (a -> c)."""
    return (not a or b) and (a or b) and (not a or c)

def worlds():
    for values in itertools.product([False, True], repeat=len(VARIABLES)):
        yield dict(zip(VARIABLES, values))

def world_weight(world, w):
    return math.prod(w[v] if value else 1 - w[v] for v, value in world.items())

MODELS = [i for i in worlds() if F(**i)]

def brute_force_prob(w, evidence=None):
    """Pr(F | evidence), with evidence given as {variable: value}."""
    evidence = evidence or {}
    joint = sum(world_weight(i, w) for i in MODELS if all(i[v] == val for v, val in evidence.items()))
    return joint / math.prod(w[v] if val else 1 - w[v] for v, val in evidence.items())

# Symbolic expressions

With `symbolic=True`, each weight is replaced by a symbol named after its variable, and the circuit is evaluated over SymPy expressions instead of numbers. For `Prob`, the result is the exact polynomial computed by weighted model counting:

In [2]:
prob_expr = solver.query(mode="Prob", symbolic=True).inference
prob_expr

a*b*c + b*(1 - a)

This is the WMC formula from the solver tutorial, a sum over the models of $F$ of products of $w(v)$ and $1 - w(v)$, which we can also build directly from the truth table and compare:

$$\text{WMC}(F ; w) := \sum_{i \models F} \prod_{i \models v} w(v) \cdot \prod_{i \models \neg v} (1 - w(v))$$

In [3]:
symbols = {s.name: s for s in prob_expr.free_symbols}

truth_table_poly = sum(
    math.prod(symbols[v] if value else 1 - symbols[v] for v, value in i.items())
    for i in MODELS
)
assert sympy.expand(prob_expr - truth_table_poly) == 0
truth_table_poly

a*b*c + b*c*(1 - a) + b*(1 - a)*(1 - c)

The circuit computes a factorized form of this polynomial, which is how it avoids enumerating all models.

#### Substituting values

Substituting the weights gives back the numerical result. The symbols are created as positive reals (`sympy.Symbol("a", positive=True, real=True)`), so it is easiest to substitute using the expression's own `free_symbols`:

In [4]:
def evaluate(expr, w):
    """Substitutes the weights `w` (a dict from variable names to values) into `expr`."""
    return float(sympy.sympify(expr).subs({s: w[s.name] for s in sympy.sympify(expr).free_symbols}))

assert math.isclose(evaluate(prob_expr, WEIGHTS), solver.query(mode="Prob").inference.item(), rel_tol=1e-6)
evaluate(prob_expr, WEIGHTS)

0.25984

Since the expression is exact, it can be evaluated at any weights without running the solver again, e.g., by turning it into a Python function with `sympy.lambdify`:

In [5]:
a, b, c = (symbols[v] for v in VARIABLES)
prob_fn = sympy.lambdify([a, b, c], prob_expr)

other_weights = {"a": 0.2, "b": 0.5, "c": 0.7}
assert math.isclose(prob_fn(0.2, 0.5, 0.7), brute_force_prob(other_weights))
prob_fn(0.2, 0.5, 0.7)

0.47000000000000003

#### Other inferences

Every inference mode can be computed symbolically. For instance, `Viterbi` gives an expression with nested maxima, one for each choice point in the circuit, and `Entropy` gives the closed form of $H(I \mid F) = \log Z - S/Z$:

In [8]:
for mode in ["LogProb", "Viterbi", "Fuzzy", "Entropy"]:
    expr = solver.query(mode=mode, symbolic=True).inference
    print(f"{mode}: {expr}\n")
    #print(expr)
    assert math.isclose(evaluate(expr, WEIGHTS), solver.query(mode=mode).inference.item(), rel_tol=1e-5)

LogProb: log(a*b*c + b*(1 - a))

Viterbi: Max(a*b*c, b*(1 - a)*Max(c, 1 - c))

Fuzzy: Min(b, Max(Min(a, c), Min(1 - a, Max(c, 1 - c))))

Entropy: log(a*b*c + b*(1 - a)) - (a*b*c*log(a) + a*(b*c*log(b) + b*c*log(c)) + b*(1 - a)*(c*log(c) + (1 - c)*log(1 - c)) + b*(1 - a)*log(b) + b*(1 - a)*log(1 - a))/(a*b*c + b*(1 - a))



Evidence works symbolically too. For example, given $a$, our formula reduces to $b \land c$, so $Pr(F \mid a) = b c$:

In [8]:
solver.query(evidence="a", symbolic=True).inference

b*c

#### Boolean formulas

The `Formula` mode evaluates the circuit in the *Boolean* semiring $(\{\text{formulas}\}, \lor, \land, \bot, \top)$, which gives back a Boolean formula equivalent to $F$ (the conjunction of `FORMULAS`):

In [9]:
formula = solver.query(mode="Formula").inference
print(formula)

## check equivalence with F
a_, b_, c_ = sympy.symbols("a b c")
F_sympy = sympy.And(sympy.Implies(a_, b_), sympy.Or(a_, b_), sympy.Implies(a_, c_))
formula = formula.subs({s: sympy.Symbol(s.name) for s in formula.free_symbols})
assert sympy.satisfiable(sympy.Not(sympy.Equivalent(formula, F_sympy))) is False

(b & ~a) | (a & b & c)


#### SymEngine

For large formulas, symbolic computation with SymPy can be slow. [SymEngine](https://github.com/symengine/symengine.py) is a much faster C++ implementation, selected with `symbolic_backend="symengine"`. Its expressions can be converted to SymPy with `sympy.sympify`:

In [10]:
fast_expr = solver.query(mode="Prob", symbolic=True, symbolic_backend="symengine").inference
print(type(fast_expr))
assert sympy.expand(sympy.sympify(fast_expr) - prob_expr.subs({s: sympy.Symbol(s.name) for s in prob_expr.free_symbols})) == 0
fast_expr

<class 'symengine.lib.symengine_wrapper.Add'>


b*(1 - a) + a*b*c

# Gradients

Since inference is a differentiable function of the weights, we can compute its **gradient**: how the result changes as each weight changes. By default, gradients are computed by PyTorch's automatic differentiation, ordered as in `solver.weight_names`:

In [11]:
grad = solver.gradient(mode="Prob", key="weights")
dict(zip(solver.weight_names, grad.tolist()))

{'a': -0.42239999771118164, 'b': 0.406000018119812, 'c': 0.5759999752044678}

These match *finite differences*, i.e., nudging each weight by a small $\epsilon$ in the brute-force computation:

In [12]:
def finite_differences(fn, w, eps=1e-6):
    """Central finite differences of `fn(w)` for each weight."""
    return [
        (fn({**w, v: w[v] + eps}) - fn({**w, v: w[v] - eps})) / (2 * eps)
        for v in VARIABLES
    ]

fd = finite_differences(brute_force_prob, WEIGHTS)
assert all(math.isclose(g, d, abs_tol=1e-5) for g, d in zip(grad.tolist(), fd))
fd

[-0.422400000005263, 0.40600000000834413, 0.5759999999743748]

#### Symbolic gradients

With `symbolic=True`, the gradient is a vector of exact partial derivatives:

In [13]:
sym_grad = solver.gradient(mode="Prob", symbolic=True, key="weights")
sym_grad

Matrix([
[    b*c - b],
[a*c - a + 1],
[        a*b]])

In [14]:
assert all(math.isclose(evaluate(g, WEIGHTS), t, rel_tol=1e-5) for g, t in zip(sym_grad, grad.tolist()))
[evaluate(g, WEIGHTS) for g in sym_grad]

[-0.4224, 0.406, 0.5760000000000001]

#### What do these gradients mean?

Weighted model counting is *multilinear*: conditioning on each value of a variable $x$ gives

$$Pr(F) = w(x)\, Pr(F \mid x) + (1 - w(x))\, Pr(F \mid \neg x),$$

so the gradient for $x$ is the difference between the two conditional probabilities:

$$\frac{\partial Pr(F)}{\partial w(x)} = Pr(F \mid x) - Pr(F \mid \neg x).$$

That is, how much more likely $F$ becomes when $x$ is true rather than false. We can check this numerically and symbolically with evidence queries:

In [15]:
for i, x in enumerate(solver.weight_names):
    difference = solver.query(evidence=x).inference - solver.query(evidence=f"~{x}").inference
    assert math.isclose(difference.item(), grad[i].item(), rel_tol=1e-5)

    sym_difference = (
        solver.query(evidence=x, symbolic=True).inference
        - solver.query(evidence=f"~{x}", symbolic=True).inference
    )
    assert sympy.simplify(sym_difference - sym_grad[i]) == 0
    print(f"dPr(F)/dw({x}) = Pr(F | {x}) - Pr(F | ~{x}) = {sympy.simplify(sym_difference)}")

dPr(F)/dw(a) = Pr(F | a) - Pr(F | ~a) = b*(c - 1)
dPr(F)/dw(b) = Pr(F | b) - Pr(F | ~b) = a*c - a + 1
dPr(F)/dw(c) = Pr(F | c) - Pr(F | ~c) = a*b


For example, the gradient for $a$ is negative: $F$ requires $c$ whenever $a$ is true, so making $a$ more likely makes $F$ *less* likely (since $w(c) < 1$).

#### Gradients of other inferences

Gradients are available for every mode. Some have simple interpretations:

- **LogProb**: by the chain rule, $\nabla \log Pr(F) = \nabla Pr(F) / Pr(F)$.
- **Viterbi**: the score of the best model is a product of its literal weights, so only the best model contributes: the gradient for $v$ is $\text{score} / w(v)$ if $v$ is true in the best model, and $-\text{score} / (1 - w(v))$ otherwise.
- **Fuzzy**: the result is a single literal weight (the minimum literal of the best model), so the gradient is one-hot on that variable.

In [16]:
prob = solver.query(mode="Prob").inference
log_grad = solver.gradient(mode="LogProb", key="weights")
assert torch.allclose(log_grad, grad / prob)

score = solver.query(mode="Viterbi").inference.item()
best = solver.query(mode="ViterbiAssignment").variables
expected = [score / WEIGHTS[v] if best[v] else -score / (1 - WEIGHTS[v]) for v in solver.weight_names]
viterbi_grad = solver.gradient(mode="Viterbi", key="weights")
assert all(math.isclose(g, e, rel_tol=1e-5) for g, e in zip(viterbi_grad.tolist(), expected))

print("LogProb:", log_grad)
print("Viterbi:", viterbi_grad, "best model:", best)
print("Fuzzy:  ", solver.gradient(mode="Fuzzy", key="weights"))
print("Entropy:", solver.gradient(mode="Entropy", key="weights"))

LogProb: tensor([-1.6256,  1.5625,  2.2167])
Viterbi: tensor([0.2176, 0.3060, 0.5760]) best model: {'a': True, 'c': True, 'b': True}
Fuzzy:   tensor([0., 0., 1.])
Entropy: tensor([-3.6291e+00,  2.3842e-07, -7.9728e-01])


Symbolic gradients work for these modes too (`Viterbi` and `Fuzzy` give piecewise expressions with `Heaviside` step functions, which require the SymPy backend):

In [17]:
solver.gradient(mode="LogProb", symbolic=True, key="weights")

Matrix([
[    (b*c - b)/(a*b*c + b*(1 - a))],
[(a*c - a + 1)/(a*b*c + b*(1 - a))],
[          a*b/(a*b*c + b*(1 - a))]])

#### Boolean gradients

There is also a purely logical counterpart of the gradient. The *Boolean derivative* of $F$ with respect to $x$ is

$$\frac{\partial F}{\partial x} = F|_{x} \oplus F|_{\neg x},$$

where $F|_x$ is $F$ with $x$ set to true and $\oplus$ is exclusive or. It describes the worlds in which flipping $x$ changes the truth value of $F$, i.e., where $x$ *matters*:

In [18]:
boolean_grad = solver.gradient(boolean=True, simplify=True, key="weights")
dict(zip(solver.weight_names, boolean_grad))

{'a': b & ~c, 'b': c | ~a, 'c': a & b}

For example, $a$ matters exactly when $b \land \neg c$: then $F$ holds if $a$ is false but fails if $a$ is true. We can check each derivative against the truth table by flipping variables:

In [19]:
for x, derivative in zip(solver.weight_names, boolean_grad):
    derivative = derivative.subs({s: sympy.Symbol(s.name) for s in derivative.free_symbols})
    for i in worlds():
        flips_F = F(**{**i, x: True}) != F(**{**i, x: False})
        assert bool(derivative.subs({sympy.Symbol(v): val for v, val in i.items()})) == flips_F

# Learning weights

Gradients let us *learn* weights. The `as_function` method returns inference as a function of the weight vector (ordered as in `solver.weight_names`), which can be used directly with PyTorch optimizers.

As a simple example, we learn weights that make $F$ as likely as possible, i.e., maximize $\log Pr(F)$. To keep the weights in $[0, 1]$ we optimize unconstrained *logits* and map them through a sigmoid:

In [20]:
log_prob = solver.as_function(mode="LogProb")

logits = torch.zeros(len(solver.weight_names), requires_grad=True)  # all weights start at 0.5
optimizer = torch.optim.Adam([logits], lr=0.1)

for step in range(101):
    loss = -log_prob(torch.sigmoid(logits))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        weights = dict(zip(solver.weight_names, torch.sigmoid(logits).tolist()))
        print(f"step {step:3d}  Pr(F) = {math.exp(-loss.item()):.3f}  weights = { {k: round(v, 3) for k, v in weights.items()} }")

step   0  Pr(F) = 0.375  weights = {'a': 0.475, 'b': 0.525, 'c': 0.525}
step  25  Pr(F) = 0.883  weights = {'a': 0.119, 'b': 0.903, 'c': 0.881}
step  50  Pr(F) = 0.958  weights = {'a': 0.067, 'b': 0.964, 'c': 0.933}
step  75  Pr(F) = 0.975  weights = {'a': 0.053, 'b': 0.978, 'c': 0.947}
step 100  Pr(F) = 0.982  weights = {'a': 0.046, 'b': 0.984, 'c': 0.954}


The weights move towards one of the models of $F$ (here $\neg a, b, c$), making it nearly certain. Note that this changes only the copy of the weights passed to `log_prob`; the solver's own weights are unchanged:

In [21]:
solver.weights

PytorchWeights(a=0.9, b=0.64, c=0.34)

## Next steps

- **Tensorization and JAX** (`tensorization_and_jax.ipynb`): compiling these computations for fast, batched evaluation.